---
title: Глава 5. Запросы к нескольким таблицам
subtitle: SQL Lab in JupyterLab
# license: CC-BY-4.0
github: https://github.com/magus1968/learning-sql
subject: Technical Portfolio
venue: GitHub & GitVerse Pages
# abstract: |
#   В последнем запросе главы, в разделе _Этот таинственный null_, увидим причину, по которой для усечения строк в дальнейшем будет использоваться Pandas.
authors:
  - name: Alex Smirnov
    email: a@smirnovs.pro
    corresponding: true
    affiliations: Data & BI Analyst
date: 2026-08-10
abbreviations:
    MyST: Markedly Structured Text
    Jupyter Book: Build static Web-books
    JupySQL: Run & highlight SQL in Jupyter
---

In [1]:
from sqlalchemy import create_engine
from sqlalchemy.engine import URL


connection_url = URL.create(
    drivername="mysql+pymysql",
    host="localhost",
    port=3306,
    database="sakila",
    username="root",
    password="********",
)

engine = create_engine(connection_url)

%load_ext sql

%config SqlMagic.displaylimit = 20
%config SqlMagic.displaycon = False
# %config SqlMagic.feedback = False

%sql engine

print("SQLAlchemy - подключение создано")
print("JupySQL - успешно подключен через SQLAlchemy Engine!")

SQLAlchemy - подключение создано
JupySQL - успешно подключен через SQLAlchemy Engine!


Еще в главе 2 было продемонстрировано, как  связанные концепции разбиваются на отдельные части с помощью процесса, известного как нормализация. Конечным результатом этого упражнения были две таблицы: person и favorite_food.

Если вы хотите создать единый отчет с указанием имени, адреса *и* любимой еды человека, вам понадобится механизм, который снова соберет данные из этих двух таблиц. Этот механизм известен как *соединение* (join), и в этой главе основное внимание уделяется простейшему и наиболее распространенному соединению – _**внутреннему**_ (`inner join`).

В главе 10 демонстрируются все типы соединений.

## Что такое соединение

Запросы к одной таблице – это, конечно, не редкость. Но вы обраружите, что для большинства ваших запросов потребуется две, три или даже больше таблиц. 

Для иллюстрации рассмотрим определения таблиц **customer** и **address**, а затем определим запрос, который извлекает данные из обеих таблиц:

In [2]:
%%sql
DESC customer;

9 rows affected.

Field,Type,Null,Key,Default,Extra
customer_id,smallint unsigned,NO,PRI,None,auto_increment
store_id,tinyint unsigned,NO,MUL,None,
first_name,varchar(45),NO,,None,
last_name,varchar(45),NO,MUL,None,
email,varchar(50),YES,,None,
address_id,smallint unsigned,NO,MUL,None,
active,tinyint(1),NO,,1,
create_date,datetime,NO,,None,
last_update,timestamp,YES,,CURRENT_TIMESTAMP,DEFAULT_GENERATED on update CURRENT_TIMESTAMP


In [4]:
%%sql
DESC address;

9 rows affected.

Field,Type,Null,Key,Default,Extra
address_id,smallint unsigned,NO,PRI,None,auto_increment
address,varchar(50),NO,,None,
address2,varchar(50),YES,,None,
district,varchar(20),NO,,None,
city_id,smallint unsigned,NO,MUL,None,
postal_code,varchar(10),YES,,None,
phone,varchar(20),NO,,None,
location,geometry,NO,MUL,None,
last_update,timestamp,NO,,CURRENT_TIMESTAMP,DEFAULT_GENERATED on update CURRENT_TIMESTAMP


Допустим, вы хотите получить имя и фамилию каждого клиента, а также его почтовый адрес. Таким образом, ваш запрос должен будет получить столбцы **customer\.first_name**, **customer\.last_name** и **address\.address**.

Но как получить данные из обеих таблиц в одном запросе? Ответ кроется в столбце **customer\.address_id**, содержащем идентификатор записи клиента в таблице **address** (более формально – столбец **customer.adderss_id** является *внешним ключом* к таблице **address**). Запрос, который вы вскоре увидите, предписывает серверу использовать столбец **customer.adderss_id** в качестве *транспорта* между таблицами **customer** и **address**, что позволяет включать в результирующий набор запроса столбцы из обеих таблиц. Этот тип операции известе как *соединение* (`join`).

In [5]:
import pandas as pd
print(pd.__version__)

3.0.5


In [44]:
%config SqlMagic.autopandas = True

### Декартово произведение

Самый простой способ начать решать поставленную задачу – поместить таблицы **customer** и **address** в предложение `from` запроса и посмотреть что получится.

Вот запрос, который извлекает имена и фамилии клиентов вместе с почтовым адресом; с предложением `from`, указывающим обе таблицы разделенные ключевым словом `JOIN`:

In [7]:
%%sql
SELECT c.first_name, c.last_name, a.address
FROM customer c
    JOIN address a;

361197 rows affected.

,first_name,last_name,address
0,AUSTIN,CINTRON,47 MySakila Drive
1,WADE,DELVALLE,47 MySakila Drive
2,FREDDIE,DUGGAN,47 MySakila Drive
3,ENRIQUE,FORSYTHE,47 MySakila Drive
4,TERRENCE,GUNDERSON,47 MySakila Drive
...,...,...,...
361192,ELIZABETH,BROWN,1325 Fukuyama Street
361193,BARBARA,JONES,1325 Fukuyama Street
361194,LINDA,WILLIAMS,1325 Fukuyama Street
361195,PATRICIA,JOHNSON,1325 Fukuyama Street


Гм... Всего имеется 599 клиентов, а в таблице **address** 603 строки. Так откуда же в результирующем набора 361 197 строк? Присмотревшись, можно увидеть, что многие клиенты имеют один и тот же адрес. Поскольку в запросе не указано *как именно* должны быть соединены две таблицы, сервер базы данных сгенерировал *декартово произведение*, которое представляет собой *все* возможные сочетания записей из двух таблиц (599 клиентов \* 603 адреса = 361 197 сочетаний).

Этот тип соединения известен как _**перекрестное** соединение_ (`cross join`) и используется крайне редко (по крайней мере, преднамеренно). Перекрестные соединения – один из типов соединений, которые рассмотрим в главе 10.

### Внутренние соединения

Чтобы изменить предыдущий запрос так, чтобы для каждого клиента возвращалась только одна строка, нужно правильно описать, как именно связаны эти две таблицы.

Ранее я сообщил, что столбец **customer.address_id** служит связующим звеном между двумя таблицами. Поэтому необходимо добавить эту информацию в подпредложение `on` предложения `from`:

In [11]:
%%sql
SELECT c.first_name, c.last_name, a.address
FROM customer c
    JOIN address a
        ON c.address_id = a.address_id;

599 rows affected.

,first_name,last_name,address
0,MARY,SMITH,1913 Hanoi Way
1,PATRICIA,JOHNSON,1121 Loja Avenue
2,LINDA,WILLIAMS,692 Joliet Street
3,BARBARA,JONES,1566 Inegl Manor
4,ELIZABETH,BROWN,53 Idfu Parkway
...,...,...,...
594,TERRENCE,GUNDERSON,844 Bucuresti Place
595,ENRIQUE,FORSYTHE,1101 Bucuresti Boulevard
596,FREDDIE,DUGGAN,1103 Quilmes Boulevard
597,WADE,DELVALLE,1331 Usak Boulevard


Теперь вместо 361 197 строк у нас имеются ожидаемые 599 строк – благодаря добавлению подпредложения `on`, которое указывает серверу необходимость соединения таблиц **customer** и **address**, используя столбец **address_id** для перехода от одной таблицы к другой.

Например, строка 'MARY SMITH' в таблице **customer** содержит значение 5 в столбце **address_id** (в примере не показано). Сервер использует это значение для поиска в таблице **address** строки, имеющей значение 5 в столбце **address_id**, а затем извлекает значение '1913 Hanoi Way' из столбца **address** в этой строке.

:::{important}
Если значение имеется в столбце **address_id** одной таблицы, но *отсутствует* в другой, то соединение для строк, содержащих это значение, *не выполняется* и эти строки *исключаются* из результирующего набора. Этот тип соединения известен как _**внутреннее соединение**_ и является наиболее часто используемым типом соединения.

В качестве примера: если строка в таблице **customer** имеет значение 999 в столбце **address_id**, а в таблице **address** нет строки со значением 999 в столбце **address_id**, то эта строка клиента не будет включена в результирующий набор.
:::

Если же вы хотите включить все строки из одной таблицы, независимо от того, имеется ли соответствующее значение в другой, вам необходимо указать _**внешнее соединение**_, но мы отложим рассмотрение этого типа соединения до главы 10.

---

В предыдущем примере я не указывал в предложении `from` какой тип соединения использовать. Однако, чтобы соединить две таблицы с использованием *внутреннего соединения*, необходимо **явно** указать это в своем предложении `from`. Вот тот же пример с добавлением типа соединения – обратите внимание на ключевое слово `inner`:

In [12]:
%%sql
SELECT c.first_name, c.last_name, a.address
FROM customer c
    INNER JOIN address a
        ON c.address_id = a.address_id;

599 rows affected.

,first_name,last_name,address
0,MARY,SMITH,1913 Hanoi Way
1,PATRICIA,JOHNSON,1121 Loja Avenue
2,LINDA,WILLIAMS,692 Joliet Street
3,BARBARA,JONES,1566 Inegl Manor
4,ELIZABETH,BROWN,53 Idfu Parkway
...,...,...,...
594,TERRENCE,GUNDERSON,844 Bucuresti Place
595,ENRIQUE,FORSYTHE,1101 Bucuresti Boulevard
596,FREDDIE,DUGGAN,1103 Quilmes Boulevard
597,WADE,DELVALLE,1331 Usak Boulevard


Если вы не укажите тип соединения, сервер по умолчанию выполнит *внутреннее соединение*. Однако, как вы увидите далее в этой книге, существует несколько типов соединений. Поэтому лучше иметь привычку **указывать точный тип нужного вам соединения**, что будет добрым делом для других людей, которые могут использовать или поддерживать ваши запросы в будущем.

---

Если имена столбцов, используемых для соединения двух таблиц, **идентичны** (как это было в предыдущем запросе), вместо подпредложения `on` можно использовать подпредложение `using`:

In [ ]:
%%sql
SELECT c.first_name, c.last_name, a.address
FROM customer c 
    INNER JOIN address a
        USING (address_id);

:::{note}
Поскольку `using` – это сокращенная запись, которую можно использовать только в определенной ситуации, я предпочитаю **всегда использовать** подпредложение `on`, чтобы избежать путаницы.
:::

### Синтаксис соединения ANSI

In [10]:
%%sql
SELECT c.first_name, c.last_name, a.address
FROM customer c, address a
WHERE c.address_id = a.address_id;

599 rows affected.

,first_name,last_name,address
0,MARY,SMITH,1913 Hanoi Way
1,PATRICIA,JOHNSON,1121 Loja Avenue
2,LINDA,WILLIAMS,692 Joliet Street
3,BARBARA,JONES,1566 Inegl Manor
4,ELIZABETH,BROWN,53 Idfu Parkway
...,...,...,...
594,TERRENCE,GUNDERSON,844 Bucuresti Place
595,ENRIQUE,FORSYTHE,1101 Bucuresti Boulevard
596,FREDDIE,DUGGAN,1103 Quilmes Boulevard
597,WADE,DELVALLE,1331 Usak Boulevard


Преимущество синтаксиса соединения `SQL92` проще увидеть для сложных запросов, которые включают как условия соединения, так и условия фильтрации.

Старый синтаксис соединения ANSI:

In [12]:
%%sql
SELECT c.first_name, c.last_name, a.address
FROM customer c, address a
WHERE c.address_id = a.address_id
  AND a.postal_code = '52137';

2 rows affected.

,first_name,last_name,address
0,JAMES,GANNON,1635 Kuwana Boulevard
1,FREDDIE,DUGGAN,1103 Quilmes Boulevard


Запрос с использованием синтаксиса соединения `SQL92`:

In [13]:
%%sql
SELECT c.first_name, c.last_name, a.address
FROM customer c
    INNER JOIN address a ON c.address_id = a.address_id
WHERE a.postal_code = '52137';

2 rows affected.

,first_name,last_name,address
0,JAMES,GANNON,1635 Kuwana Boulevard
1,FREDDIE,DUGGAN,1103 Quilmes Boulevard


## Соединение трех и более таблиц

Соединение трех таблиц аналогично соединению двух таблиц, но с одним небольшим отличием. При соединении двух таблиц имеются две таблицы и один тип соединения в предложении `from`, а также одно подпредложение `on` определяющее как таблицы соединяются.

При соединении трех таблиц, в предложении `from` есть три таблицы, два типа соединения и два подпредложения `on`.

Чтобы проиллюстрировать это, давайте изменим предыдущий запрос, чтобы возвращать город клиента, а не его почтовый адрес. Однако название города в таблице **address** не хранится, а доступно через внешний ключ к таблице **city**. Вот так выглядят определения таблиц:

In [15]:
%%sql
DESC address;

9 rows affected.

Field,Type,Null,Key,Default,Extra
address_id,smallint unsigned,NO,PRI,None,auto_increment
address,varchar(50),NO,,None,
address2,varchar(50),YES,,None,
district,varchar(20),NO,,None,
city_id,smallint unsigned,NO,MUL,None,
postal_code,varchar(10),YES,,None,
phone,varchar(20),NO,,None,
location,geometry,NO,MUL,None,
last_update,timestamp,NO,,CURRENT_TIMESTAMP,DEFAULT_GENERATED on update CURRENT_TIMESTAMP


In [16]:
%%sql
DESC city;

4 rows affected.

Field,Type,Null,Key,Default,Extra
city_id,smallint unsigned,NO,PRI,None,auto_increment
city,varchar(50),NO,,None,
country_id,smallint unsigned,NO,MUL,None,
last_update,timestamp,NO,,CURRENT_TIMESTAMP,DEFAULT_GENERATED on update CURRENT_TIMESTAMP


Чтобы показать город каждого клиента, необходимо перейти от таблицы **customer** к таблице **address** (используя столбец **address_id**), а затем – от таблицы **address** к таблице **city** с использованием столбца **city_id**.

Запрос будет выглядеть следующим образом:

In [35]:
%%sql
SELECT c.first_name, c.last_name, ct.city
FROM customer c
    INNER JOIN address a
        ON c.address_id = a.address_id
    INNER JOIN city ct
        ON a.city_id = ct.city_id;

599 rows affected.

,first_name,last_name,city
0,MARY,SMITH,Sasebo
1,PATRICIA,JOHNSON,San Bernardino
2,LINDA,WILLIAMS,Athenai
3,BARBARA,JONES,Myingyan
4,ELIZABETH,BROWN,Nantou
...,...,...,...
594,TERRENCE,GUNDERSON,Jinzhou
595,ENRIQUE,FORSYTHE,Patras
596,FREDDIE,DUGGAN,Sullana
597,WADE,DELVALLE,Lausanne


Для этого запроса используются три таблицы, два соединения и два подпредложения `on` в `from`, так что все стало выглядеть более запутанным. На первый взгляд может показаться, что порядок, в котором таблицы появляются в предложении `from` важен. Но если вы измените порядок таблиц, то получите точно такие же результаты.

Все три приведенных ниже варианта запроса возвращают одни и те же результаты:

In [ ]:
%%sql
SELECT c.first_name, c.last_name, ct.city
FROM customer c
    INNER JOIN address a
        ON c.address_id = a.address_id
    INNER JOIN city ct
        ON a.city_id = ct.city_id;

In [ ]:
%%sql
SELECT c.first_name, c.last_name, ct.city
FROM city ct
    INNER JOIN address a
        ON a.city_id = ct.city_id
    INNER JOIN customer c
        ON c.address_id = a.address_id;

In [ ]:
%%sql
SELECT c.first_name, c.last_name, ct.city
FROM address a
    INNER JOIN city ct
        ON a.city_id = ct.city_id
    INNER JOIN customer c
        ON c.address_id = a.address_id;

Единственное различие, которое можно увидеть – это порядок, в котором возвращаются строки, поскольку в запросах нет предложения `order by`, указывающего как должны быть упорядочены результаты.

:::{tip} Имеет ли значение порядок соединения
:class: dropdown
:open: true
Если вы не понимаете, почему все три версии запроса **customer**/**address**/**city** дают одни и те же результаты, вспомните, что **SQL не является процедурным языком**.

Это означает, что вы описываете _**что**_ хотите получить и какие объекты базы данных должны быть вовлечены в запрос. Но _**как**_ лучше всего выполнить ваш запрос – определяет сервер базы данных.

Используя статистику, собранную из объектов вашей базы данных, сервер должен выбрать одну из трех таблиц в качестве отправной точки (выбранная таблица в дальнейшем называется *ведущей таблицей* (driving table)), а затем решить в каком порядке соединять с ней оставшиеся таблицы. Следовательно **порядок, в котором таблицы появляются в вашем предложении `from` значения не имеет**.

---

Однако, если вы считаете, что таблицы в вашем запросе всегда следует соединять в опредененном порядке, можете разместить таблицы в желаемом порядке, а затем указать ключевое слово `stright_join`.

Например, чтобы указать серверу MySQL использовать в качестве ведущей таблицы **city**, а затем присоединить таблицы **address** и **customer**, можно сделать следующее:
:::

In [34]:
%%sql
SELECT STRAIGHT_JOIN c.first_name, c.last_name, ct.city
FROM city ct
    INNER JOIN address a
        ON a.city_id = ct.city_id
    INNER JOIN customer c
        ON c.address_id = a.address_id;

599 rows affected.

,first_name,last_name,city
0,JULIE,SANCHEZ,A Coruña (La Coruña)
1,PEGGY,MYERS,Abha
2,TOM,MILNER,Abu Dhabi
3,GLEN,TALBERT,Acuña
4,LARRY,THRASHER,Adana
...,...,...,...
594,CONSTANCE,REID,Zaria
595,JACK,FOUST,Zeleznogorsk
596,BYRON,BOX,Zhezqazghan
597,GUY,BROWNLEE,Zhoushan


### Использование подзапросов в качестве таблиц

Вы уже видели несколько примеров запросов, включающих несколько таблиц. Но стоит упомянуть еще один вариант: что делать, если некоторые из наборов данных сгенерированы подзапросами? *Подзапросы* – это основная тема главы 9, но я уже знакомил вас с этой концепцией в предыдущей главе.

Следующий запрос соединяет таблицу **customer** с *подзапросом* к таблицам **address** и **city**:

In [36]:
%%sql
SELECT c.first_name, c.last_name, addr.address, addr.city
FROM customer c
    INNER JOIN (
        SELECT a.address_id, a.address, ct.city
        FROM address a
            INNER JOIN city ct
                ON a.city_id = ct.city_id
        WHERE a.district = 'California'
    ) AS addr
        ON c.address_id = addr.address_id;

9 rows affected.

,first_name,last_name,address,city
0,PATRICIA,JOHNSON,1121 Loja Avenue,San Bernardino
1,BETTY,WHITE,770 Bydgoszcz Avenue,Citrus Heights
2,ALICE,STEWART,1135 Izumisano Parkway,Fontana
3,ROSA,REYNOLDS,793 Cam Ranh Avenue,Lancaster
4,RENEE,LANE,533 al-Ayn Boulevard,Compton
5,KRISTIN,JOHNSTON,226 Brest Manor,Sunnyvale
6,CASSANDRA,WALTERS,920 Kumbakonam Loop,Salinas
7,JACOB,LANCE,1866 al-Qatif Avenue,El Monte
8,RENE,MCALISTER,1895 Zhezqazghan Drive,Garden Grove


Подзапрос, который начинается в строке 4 и имеет псевдоним **addr**, находит все адреса в Калифорнии. Внешний запрос соединяет результаты подзапроса с таблицей **customer**, чтобы вернуть имя, фамилию, почтовый адрес и город для всех клиентов живущих в Калифорнии.

:::{note} Примечание
:class: simple
:open: false
:icon: false
Хотя этот запрос можно было бы написать без использования подзапроса, просто соединив три таблицы, иногда применение подзапроса может быть выгодным с точки зрения производительности и/или удобочитаемости.

```sql
SELECT c.first_name, c.last_name, a.address, ct.city
FROM customer c
    INNER JOIN address a
        ON c.address_id = a.address_id
    INNER JOIN city ct
        ON a.city_id = ct.city_id
WHERE a.district = 'California';
```
:::

Один из способов визуализировать происходящее – выполнить подзапрос сам по себе и посмотреть на полученные результаты. Вот результаты *подзапроса* из предыдущего примера:

In [37]:
%%sql
SELECT a.address_id, a.address, ct.city
FROM address a
    INNER JOIN city ct
        ON a.city_id = ct.city_id
WHERE a.district = 'California';

9 rows affected.

,address_id,address,city
0,6,1121 Loja Avenue,San Bernardino
1,18,770 Bydgoszcz Avenue,Citrus Heights
2,55,1135 Izumisano Parkway,Fontana
3,116,793 Cam Ranh Avenue,Lancaster
4,186,533 al-Ayn Boulevard,Compton
5,218,226 Brest Manor,Sunnyvale
6,274,920 Kumbakonam Loop,Salinas
7,425,1866 al-Qatif Avenue,El Monte
8,599,1895 Zhezqazghan Drive,Garden Grove


Этот результирующий набор состоит из всех девяти адресов в Калифорнии. При соединении с таблицей **customer** через столбец **address_id** результирующий набор будет содержать информацию о клиентах, которым принадлежат эти адреса.

In [38]:
pd.set_option('display.max_rows', 20)

### Использование одной таблицы дважды

Соединяя несколько таблиц, можем обнаружить, что нужно соединиться с одной и той же таблицей более одного раза. В рассматриваемом нами примере базы данных, например, актеры связаны с фильмами в которых они появлялись через таблицу **film_actor**. Если хотим найти все фильмы, в которых фигурирует два конкретных актера, можем написать запрос, который соединяет таблицу **film** с таблицей **film_actor** и с таблицей **actor**:

:::{note} Таблицы  **film**, **actor** и **film_actor**
:class: dropdown simple
:open: false
:icon: false

```bash
SELECT film_id, title, release_year
FROM film;

| film_id | title            | release_year |
| ------- | ---------------- | ------------ |
| 1       | ACADEMY DINOSAUR | 2006         |
| 2       | ACE GOLDFINGER   | 2006         |
| 3       | ADAPTATION HOLES | 2006         |
| 4       | AFFAIR PREJUDICE | 2006         |
| 5       | AFRICAN EGG      | 2006         |
| ...     | ...              | ...          |
1000 rows × 3 columns
```

```bash
SELECT * FROM actor;

| actor_id | first_name | last_name    | last_update         |
| -------- | ---------- | ------------ | ------------------- |
| 1        | PENELOPE   | GUINESS      | 2006-02-15 04:34:33 |
| 2        | NICK       | WAHLBERG     | 2006-02-15 04:34:33 |
| 3        | ED         | CHASE        | 2006-02-15 04:34:33 |
| 4        | JENNIFER   | DAVIS        | 2006-02-15 04:34:33 |
| 5        | JOHNNY     | LOLLOBRIGIDA | 2006-02-15 04:34:33 |
| ...      | ...        | ...          | ...                 |
200 rows × 4 columns
```

```bash
SELECT * FROM film_actor;

| actor_id | film_id | last_update         |
| -------- | ------- | ------------------- |
| 1        | 1       | 2006-02-15 05:05:03 |
| 1        | 23      | 2006-02-15 05:05:03 |
| 1        | 25      | 2006-02-15 05:05:03 |
| 1        | 106     | 2006-02-15 05:05:03 |
| 1        | 140     | 2006-02-15 05:05:03 |
| ...      | ...     | ...                 |
5462 rows × 3 columns
```
:::

:::{tip} Продвинутый трюк: Кортежи (Tuple Comparison)
:class: simple dropdown
:open: false
:icon: false

```sql
SELECT f.title
FROM film f
    INNER JOIN film_actor fa ON f.film_id = fa.film_id
    INNER JOIN actor a ON fa.actor_id = a.actor_id
WHERE (a.first_name, a.last_name) IN (
    ('CATE', 'MCQUEEN'),
    ('CUBA', 'BIRCH')
);
```
- Запрос читается буквально как список людей.
- Если нужно будет добавить третьего актёра, просто дописываем строчку ('TOM', 'HANKS'), не раздувая дерево из скобок и операторов OR.
:::

In [39]:
%%sql
SELECT f.title
FROM film f
    INNER JOIN film_actor fa ON f.film_id = fa.film_id
    INNER JOIN actor a ON fa.actor_id = a.actor_id
WHERE
    (a.first_name = 'CATE' AND a.last_name = 'MCQUEEN')
    OR (a.first_name = 'CUBA' AND a.last_name = 'BIRCH');

54 rows affected.

,title
0,ATLANTIS CAUSE
1,BLOOD ARGONAUTS
2,COMMANDMENTS EXPRESS
3,DYNAMITE TARZAN
4,EDGE KISSING
...,...
49,TOWERS HURRICANE
50,TROJAN TOMORROW
51,VIRGIN DAISY
52,VOLCANO TEXAS


Этот запрос возвращает все фильмы в которых снимались Cate McQueen или Cuba Birch.

Предположим, что требуется получить только те фильмы, в которых появляются оба актера. Для этого нужно найти все строки в таблице **film** у которых есть две строки в таблице **film_actor**, одна из которых связана с Cate McQueen, а другая с Cuba Birch. Следовательно, требуется включить таблицы **film_actor** и **actor** дважды – каждый раз с иным псевдонимом, чтобы сервер знал на что именно мы ссылаемся в различных предложениях:

In [41]:
%%sql
SELECT f.title
FROM film f
    INNER JOIN film_actor fa1 ON f.film_id = fa1.film_id
    INNER JOIN actor a1 ON fa1.actor_id = a1.actor_id
    INNER JOIN film_actor fa2 ON f.film_id = fa2.film_id
    INNER JOIN actor a2 ON fa2.actor_id = a2.actor_id
WHERE
    (a1.first_name = 'CATE' AND a1.last_name = 'MCQUEEN')
    AND (a2.first_name = 'CUBA' AND a2.last_name = 'BIRCH');

2 rows affected.

title
BLOOD ARGONAUTS
TOWERS HURRICANE


Эти два актера снялись в 54 разных фильмах, но есть всего два фильма, в которых снялись оба актера.

Это один из примеров запроса, для которого использование псевдонимов таблиц обязательно, поскольку одни и те же таблицы используются несколько раз.

---

### Самосоединение

Вы можете не только включать одну и ту же таблицу в один и тот же запрос более одного раза, но и соединять таблицу с самой собой. Поначалу это может показаться странным, но для этого есть веские причины.

Некоторые таблицы включают в себя *самоссылающиеся внешние ключи* (self-referencing foreign key). Это означает, что в таблице имеется столбец, ссылающийся на первичный ключ в той же таблице.

Хотя образец базы данных такую связь не включает, давайте представим, что в таблице **film** есть столбец **prequel_film_id**, который указывает на родительский фильм (например, фильм *Потерянный скрипач 2* будет использовать этот столбец, чтобы указать на фильм *Потерянный скрипач* как на родительский).

Вот как выглядела бы таблица, если бы мы добавили этот дополнительный столбец:

```sql
DESC film;
```

| Field                | Type                                                                | Null | Key | Default           |
| -------------------- | ------------------------------------------------------------------- | ---- | --- | ----------------- |
| film_id              | smallint unsigned                                                   | NO   | PRI | None              |
| title                | varchar(128)                                                        | NO   | MUL | None              |
| description          | text                                                                | YES  |     | None              |
| release_year         | year                                                                | YES  |     | None              |
| language_id          | tinyint unsigned                                                    | NO   | MUL | None              |
| original_language_id | tinyint unsigned                                                    | YES  | MUL | None              |
| rental_duration      | tinyint unsigned                                                    | NO   |     | 3                 |
| rental_rate          | decimal(4,2)                                                        | NO   |     | 4.99              |
| length               | smallint unsigned                                                   | YES  |     | None              |
| replacement_cost     | decimal(5,2)                                                        | NO   |     | 19.99             |
| rating               | enum('G','PG','PG-13','R','NC-17')                                  | YES  |     | G                 |
| special_features     | set('Trailers','Commentaries','Deleted Scenes','Behind the Scenes') | YES  |     | None              |
| last_update          | timestamp                                                           | NO   |     | CURRENT_TIMESTAMP |
| prequel_film_id      | smallint(5)                                                         | YES  | MUL | NULL              |

Используя **самосоединение** можно написать запрос, в котором будут перечислены все фильмы с приквелами, включая название приквела:

```sql
SELECT f.title, f_prnt.title prequel
FROM film f
    INNER JOIN film f_prnt
        ON f_prnt.film_id = f.prequel_film_id
WHERE f.prequel_film_id IS NOT NULL;
```
```text
| title           | prequel      |
| --------------- | ------------ |
| FIDDLER LOST II | FIDDLER LOST |
```
Этот запрос соединяет таблицу **film** с самой собой с помощью внешнего ключа prequel_film_id. Псевдонимы таблицы **f** и **f_print** используются для того, чтобы было понятно, какая таблица и для какой цели используется.

---

## Упражнения

### Упражнение 5.1
Заполните пропущенные места (обозначенные как <\#>) в следующем запросе так, чтобы получить показанные результаты.

```sql
SELECT c.first_name, c.last_name, a.address, ct.city
FROM customer c
  INNER JOIN address <1>
  ON c.address_id = a.address_id
  INNER JOIN city ct
  ON a.city_id = <2>
WHERE a.district = 'California';
```
```text
| first_name | last_name | address                | city           |
| ---------- | --------- | ---------------------- | -------------- |
| PATRICIA   | JOHNSON   | 1121 Loja Avenue       | San Bernardino |
| BETTY      | WHITE     | 770 Bydgoszcz Avenue   | Citrus Heights |
| ALICE      | STEWART   | 1135 Izumisano Parkway | Fontana        |
| ROSA       | REYNOLDS  | 793 Cam Ranh Avenue    | Lancaster      |
| RENEE      | LANE      | 533 al-Ayn Boulevard   | Compton        |
| KRISTIN    | JOHNSTON  | 226 Brest Manor        | Sunnyvale      |
| CASSANDRA  | WALTERS   | 920 Kumbakonam Loop    | Salinas        |
| JACOB      | LANCE     | 1866 al-Qatif Avenue   | El Monte       |
| RENE       | MCALISTER | 1895 Zhezqazghan Drive | Garden Grove   |
```

In [19]:
%%sql
SELECT c.first_name, c.last_name, a.address, ct.city
FROM customer c
  INNER JOIN address a
  ON c.address_id = a.address_id
  INNER JOIN city ct
  ON a.city_id = ct.city_id
WHERE a.district = 'California';

9 rows affected.

first_name,last_name,address,city
PATRICIA,JOHNSON,1121 Loja Avenue,San Bernardino
BETTY,WHITE,770 Bydgoszcz Avenue,Citrus Heights
ALICE,STEWART,1135 Izumisano Parkway,Fontana
ROSA,REYNOLDS,793 Cam Ranh Avenue,Lancaster
RENEE,LANE,533 al-Ayn Boulevard,Compton
KRISTIN,JOHNSTON,226 Brest Manor,Sunnyvale
CASSANDRA,WALTERS,920 Kumbakonam Loop,Salinas
JACOB,LANCE,1866 al-Qatif Avenue,El Monte
RENE,MCALISTER,1895 Zhezqazghan Drive,Garden Grove



---

### Упражнение 5.2

Напишите запрос, который выводил бы названия всех фильмов, в которых играл актер с именем JOHN.

In [3]:
%config SqlMagic.displaylimit = 30

In [4]:
%%sql
SELECT f.title
FROM film f
  INNER JOIN film_actor fa
  ON f.film_id = fa.film_id
  INNER JOIN actor a
  ON fa.actor_id = a.actor_id
WHERE a.first_name = 'JOHN';

29 rows affected.

title
ALLEY EVOLUTION
BEVERLY OUTLAW
CANDLES GRAPES
CLEOPATRA DEVIL
COLOR PHILADELPHIA
CONQUERER NUTS
DAUGHTER MADIGAN
GLEAMING JAWBREAKER
GOLDMINE TYCOON
HOME PITY



---

### Упражнение 5.3

Создайте запрос, который возвращает все адреса в одном и том же городе. Вам нужно будет соединить таблицу адресов с самой собой, и каждая строка должна включать два разных адреса.

In [24]:
%%sql
-- Решение автора через декартово произведение `Cross Join`
SELECT a1.address addr1, a2.address addr2, a1.city_id
FROM address a1
  INNER JOIN address a2
WHERE a1.city_id = a2.city_id
  AND a1.address_id <> a2.address_id;

8 rows affected.

addr1,addr2,city_id
47 MySakila Drive,23 Workhaven Lane,300
28 MySQL Boulevard,1411 Lillydale Drive,576
23 Workhaven Lane,47 MySakila Drive,300
1411 Lillydale Drive,28 MySQL Boulevard,576
1497 Yuzhou Drive,548 Uruapan Street,312
587 Benguela Manor,43 Vilnius Manor,42
548 Uruapan Street,1497 Yuzhou Drive,312
43 Vilnius Manor,587 Benguela Manor,42


:::{important} Корректировка контекста задачи
Создайте запрос, который найдет **пары разных адресов**, расположенных в одном и том же городе.

**Цель запроса** – найти города, в которых зарегистрировано больше одного адреса _и показать эти адреса друг напротив друга_.
:::

:::{tip} С точки зрения современного синтаксиса **SQL92**
Правильнее и понятнее записать через `ON` – мы явно говорим базе данных: **Соединяй только те строки, у которых совпадают города** и уходим от логики _сначала соедини всё подряд, а потом отфильтруй в WHERE_.
:::

In [47]:
%%sql
-- С точки зрения современного синтаксиса SQL92
-- правильнее и понятнее записать через `ON`
SELECT a1.address addr1, a2.address addr2, a1.city_id
FROM address a1
  INNER JOIN address a2
  ON a1.city_id = a2.city_id
WHERE a1.address_id <> a2.address_id
ORDER BY a1.city_id;

8 rows affected.

addr1,addr2,city_id
587 Benguela Manor,43 Vilnius Manor,42
43 Vilnius Manor,587 Benguela Manor,42
47 MySakila Drive,23 Workhaven Lane,300
23 Workhaven Lane,47 MySakila Drive,300
1497 Yuzhou Drive,548 Uruapan Street,312
548 Uruapan Street,1497 Yuzhou Drive,312
28 MySQL Boulevard,1411 Lillydale Drive,576
1411 Lillydale Drive,28 MySQL Boulevard,576


:::{hint} Как сделать этот запрос идеальным?
Чтобы избавиться от _зеркальных_ дубликатов _(когда база выводит сначала пару `A и B`, а потом `B и A`)_, в реальной разработке условие `<>` _(не равно)_ заменяют на оператор `<` _(меньше)_.

- При таком условии будет выведено `A и В`, так как ID у `A` _меньше_, чем у `B`.
- Комбо `B и A` фильтр _WHERE_ уже не пропустит, так как ID `B` будет _больше_ `А`.
:::

In [48]:
%%sql
SELECT a1.address addr1, a2.address addr2, a1.city_id
FROM address a1
  INNER JOIN address a2
  ON a1.city_id = a2.city_id
WHERE a1.address_id < a2.address_id
ORDER BY a1.city_id;

4 rows affected.

addr1,addr2,city_id
587 Benguela Manor,43 Vilnius Manor,42
47 MySakila Drive,23 Workhaven Lane,300
1497 Yuzhou Drive,548 Uruapan Street,312
28 MySQL Boulevard,1411 Lillydale Drive,576



---